# Week 3: Multi-View Reconstruction

This notebook implements the **Incremental SfM Pipeline**.
We loop through a sequence of images, adding them one by one to our 3D map using **PnP**.

In [1]:
import cv2
import numpy as np
import open3d as o3d
import os
import sys
import matplotlib.pyplot as plt

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.sfm import SfMMap
from src.reconstruction import get_intrinsic_matrix, create_point_cloud

In [2]:
# -- 1. Load Image Sequence --

# Generate the list of filenames automatically: photo1.jpeg to photo45.jpeg
# range(1, 46) generates numbers 1 to 45
IMAGE_SEQUENCE = [f'photo{i}.jpeg' for i in range(1, 46)]

data_dir = os.path.join(module_path, 'data2')
images = []

print(f"Attempting to load {len(IMAGE_SEQUENCE)} images from: {data_dir}")

for name in IMAGE_SEQUENCE:
    path = os.path.join(data_dir, name)
    img = cv2.imread(path)
    if img is not None:
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        images.append(img_rgb)
        # Optional: Print every 5th image just to show progress without clogging output
        if int(name.replace('photo', '').replace('.jpeg', '')) % 5 == 0:
            print(f"Loaded {name}...")
    else:
        print(f"Warning: Could not load {name} - Check if file exists in data/")

print(f"\nSuccessfully loaded {len(images)} images.")

Attempting to load 45 images from: /Users/maryamrizwan/Documents/GitHub/CS-436---Project/data2
Loaded photo5.jpeg...
Loaded photo10.jpeg...
Loaded photo15.jpeg...
Loaded photo20.jpeg...
Loaded photo25.jpeg...
Loaded photo30.jpeg...
Loaded photo35.jpeg...
Loaded photo40.jpeg...
Loaded photo45.jpeg...

Successfully loaded 45 images.


In [3]:
# -- 2. Initialize Map (Frame 0 & 1) --

# Estimate K (Assuming constant K for all images)
h, w = images[0].shape[:2]
K = get_intrinsic_matrix((h, w))

# Initialize our SfM Class
sfm = SfMMap(K)

# Bootstrap with first two images
sfm.initialize(images[0], images[1], lowe_ratio=0.8)

Initializing Map with first two images...
Found 2483 keypoints in img1 and 2679 in img2.
Found 2483 initial matches.
Filtered down to 664 good matches using Lowe's ratio test.
Map initialized with 319 points.


In [4]:
# -- 3. Incremental Loop (Frame 2 -> N) --

for i in range(2, len(images)):
    print(f"\nProcessing Frame {i}...")
    sfm.add_view(images[i], lowe_ratio=0.8)


Processing Frame 2...
View Added. Inliers: 25, New Points: 232, Total: 551

Processing Frame 3...
View Added. Inliers: 16, New Points: 182, Total: 733

Processing Frame 4...
View Added. Inliers: 9, New Points: 268, Total: 1001

Processing Frame 5...
View Added. Inliers: 33, New Points: 323, Total: 1324

Processing Frame 6...
View Added. Inliers: 106, New Points: 159, Total: 1483

Processing Frame 7...
View Added. Inliers: 134, New Points: 223, Total: 1706

Processing Frame 8...
View Added. Inliers: 143, New Points: 247, Total: 1953

Processing Frame 9...
PnP Failed: Geometric check failed.

Processing Frame 10...
PnP Failed: Geometric check failed.

Processing Frame 11...
PnP Failed: Geometric check failed.

Processing Frame 12...
PnP Failed: Geometric check failed.

Processing Frame 13...
PnP Failed: Geometric check failed.

Processing Frame 14...
PnP Failed: Geometric check failed.

Processing Frame 15...
PnP Failed: Geometric check failed.

Processing Frame 16...
PnP Failed: Geomet

In [9]:
# -- 4. Visualization --

points = np.array(sfm.points_3d)
colors = np.array(sfm.colors)

print(f"Final Cloud has {len(points)} points.")

# Filter outliers for cleaner visualization
mean = np.mean(points, axis=0)
std = np.std(points, axis=0)
mask = (np.abs(points - mean) < 2.5 * std).all(axis=1)

filtered_points = points[mask]
filtered_colors = colors[mask]

# Create Point Cloud
pcd = create_point_cloud(filtered_points, filtered_colors)

# --- VISUALIZE TRAJECTORY ---
camera_centers = []
camera_frustums = []

# Make axes bigger (size=2.0 or even 5.0 depending on your scene scale)
# You can adjust this number if they are still too small/large
AXIS_SIZE = 2.0 

for R, t in sfm.poses:
    # 4x4 matrix
    T = np.eye(4)
    T[:3, :3] = R
    T[:3, 3] = t.flatten()
    
    # Invert for display (Camera -> World)
    T_inv = np.linalg.inv(T)
    center = T_inv[:3, 3]
    camera_centers.append(center)
    
    axis = o3d.geometry.TriangleMesh.create_coordinate_frame(size=AXIS_SIZE, origin=[0,0,0])
    axis.transform(T_inv)
    camera_frustums.append(axis)

# Create a LineSet to connect the camera centers (The Trajectory Path)
if len(camera_centers) > 1:
    lines = []
    for i in range(len(camera_centers) - 1):
        lines.append([i, i+1])
    
    line_set = o3d.geometry.LineSet()
    line_set.points = o3d.utility.Vector3dVector(camera_centers)
    line_set.lines = o3d.utility.Vector2iVector(lines)
    # Color the path Red
    line_set.colors = o3d.utility.Vector3dVector([[1, 0, 0] for _ in lines])
    
    camera_frustums.append(line_set)

# Combine everything
geometries = [pcd] + camera_frustums

print("Opening Visualization...")
print(f"Drawing {len(filtered_points)} points and {len(camera_centers)} cameras.")
o3d.visualization.draw_geometries(geometries, window_name="Week 3: Incremental SfM")

Final Cloud has 5612 points.
Opening Visualization...
Drawing 5572 points and 32 cameras.
